# I-0 — CIFAR-100: Mish vs SpLR(Dropout = 0.10)


# SpLR (Localized Activation Residual) 

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SpLR_2d(nn.Module):
    """
    SpLR activation for Conv2d outputs.

    Formula:
        y = x + α * x * exp(-β * x^2)

    Golden Init:
        α_raw init = 0.182  → α = 2 * tanh(α_raw)  ≈ 0.36 at start
            - shape: (1, C, 1, 1)  [per-channel, broadcasts over H,W]
            - trainable, not stage-shared

        β_raw init = -1.35 → β = 0.01 + softplus(β_raw) ≈ 0.24 at start
            - scalar (one per activation block)
            - trainable, stage-shared, positive-only
    """
    def __init__(self, num_channels: int):
        super().__init__()
        # α_raw: one parameter per channel
        self.alpha_raw = nn.Parameter(torch.full((1, num_channels, 1, 1), 0.182))
        # β_raw: single scalar for the whole block
        self.beta_raw  = nn.Parameter(torch.tensor(-1.35))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        alpha = 2.0 * torch.tanh(self.alpha_raw)          # local amplitude
        beta  = 0.01 + F.softplus(self.beta_raw)          # positive width
        return x + alpha * x * torch.exp(-beta * x * x)   # identity + localized bump


class SpLR_1d(nn.Module):
    """
    SpLR activation for Linear (fully-connected) outputs.

    Same formula as SpLR_2d, but α_raw has shape (1, F)
    so it broadcasts over the batch dimension.
    """
    def __init__(self, num_features: int):
        super().__init__()
        # α_raw: one parameter per feature
        self.alpha_raw = nn.Parameter(torch.full((1, num_features), 0.182))
        # β_raw: scalar shared for this block
        self.beta_raw  = nn.Parameter(torch.tensor(-1.35))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        alpha = 2.0 * torch.tanh(self.alpha_raw)
        beta  = 0.01 + F.softplus(self.beta_raw)
        return x + alpha * x * torch.exp(-beta * x * x)



## 1. Purpose

I want to answer a single controlled question:

> If i hold the model, training regime, and dropout constant, does **SpLR** still outperform **Mish** on a harder dataset (**CIFAR-100**)?

This is **not** a “highest score” experiment.  
It is a **behavior comparison** between two activations under equal conditions.

---

## 2. Setup

### Dataset

- Dataset: **CIFAR-100**
- Train: 50,000 images (we use 45,000 train + 5,000 val)
- Test: 10,000 images
- Number of classes: 100
- Image size: 32×32 RGB

**Normalization**

- mean = `[0.5071, 0.4867, 0.4408]`
- std  = `[0.2675, 0.2565, 0.2761]`

**Augmentation**

- RandomCrop(32, padding=4)
- RandomHorizontalFlip()

### Model architecture (shared for both activations)

Simple CNN:

- Block 1: `Conv2d(3→64)` → `BatchNorm2d` → **Activation** → `Dropout2d(p=0.10)` → `MaxPool2d(2)`
- Block 2: `Conv2d(64→128)` → `BatchNorm2d` → **Activation** → `Dropout2d(p=0.10)` → `MaxPool2d(2)`
- Block 3: `Conv2d(128→256)` → `BatchNorm2d` → **Activation** → `Dropout2d(p=0.10)` → `MaxPool2d(2)`
- Flatten: `256 × 4 × 4`
- FC block: `Linear(flat→512)` → **Activation** → `Dropout(p=0.10)` → `Linear(512→100)`

The **only difference** between runs is the activation function.

### Training configuration

- Optimizer: **Adam**
- Learning rate: **1e-3**
- Loss: **CrossEntropyLoss**
- Epochs: **25**
- Seeds: **42, 1337, 777**
- Device: `cuda` if available, else `cpu`

---

## 3. Activations

### Mish (baseline)

\[
\text{mish}(x) = x \cdot \tanh(\text{softplus}(x))
\]

Implemented via `torch.nn.functional.mish`.

---

### SpLR (identity-preserving, Golden Init)

Activation:

\[
y = x + \alpha \, x \, e^{-\beta x^2}
\]

Implementation details:

- **α_raw init** = 0.182  
  - α = \( 2 \tanh(\alpha_\text{raw}) \) ≈ 0.36 at init  
  - trainable, **per-channel** for conv / per-feature for linear
- **β_raw init** = −1.35  
  - β = \( 0.01 + \text{softplus}(\beta_\text{raw}) \) ≈ 0.24 at init  
  - trainable, **scalar per block** (stage-shared)
- β is positive-only.

We use:
- `SpLR_2d` after conv layers
- `SpLR_1d` after the first linear layer

Dropout (p = 0.10) is applied after each activation.

---

## 4. Results

### Raw per-seed results

**Mish**

| seed | best_val_acc | best_test_acc_at_best_val |
|------|--------------|---------------------------|
| 42   | 0.5328       | 0.5608                    |
| 1337 | 0.5302       | 0.5626                    |
| 777  | 0.5344       | 0.5617                    |

**SpLR**

| seed | best_val_acc | best_test_acc_at_best_val |
|------|--------------|---------------------------|
| 42   | 0.5476       | 0.5871                    |
| 1337 | 0.5406       | 0.5848                    |
| 777  | 0.5468       | 0.5829                    |

### Summary statistics

| activation | mean_val_acc | std_val_acc | mean_test_acc | std_test_acc |
|-----------|--------------|-------------|---------------|--------------|
| mish      | 0.5325       | 0.0021      | 0.5617        | 0.0009       |
| SpLR      | **0.5450**   | 0.0038      | **0.5849**    | 0.0021       |

---

## 5. Interpretation

**Observation**

- SpLR outperforms Mish on both validation and test.
- The gap is consistent across all seeds.
- Variance is small, so the improvement is unlikely to be random noise.

**Hypothesized reasons (not proven yet)**

1. **Identity-preserving behavior**  
   SpLR keeps the raw signal (`+ x`) and adds a *local* correction, which helps gradients flow in deeper stacks.

2. **Localized shaping**  
   The Gaussian term \( e^{-\beta x^2} \) focuses the correction on useful magnitude ranges instead of reshaping all values like Mish.

3. **Reduced need for strong regularization**  
   With dropout \( p = 0.10 \), SpLR generalizes well. Heavier dropout previously showed harm, suggesting SpLR is partially self-regularizing.

4. **Fine-grained task**  
   CIFAR-100 needs subtle feature distinctions. SpLR’s signal-preserving design may help retain mid-range feature information that Mish slightly smooths out.

---

## 6. What we are NOT claiming

We are **not** claiming:

- that SpLR is universally better than Mish on all architectures and tasks;
- that dropout = 0.10 is always optimal;
- that this single benchmark is sufficient to declare superiority.

This is **one controlled comparison** on CIFAR-100 with a specific CNN and training regime.

---

## 7. Follow-up experiments (planned)

Recommended next steps:

1. **No-dropout comparison**  
   SpLR vs Mish with `dropout = 0.0` to see if SpLR still wins and how much each overfits.

2. **Different dropout levels**  
   Example: Mish with `p = 0.25` vs SpLR with `p = 0.10` to test whether Mish needs stronger regularization.

3. **Weight decay sweeps (L2 regularization)**  
   Check if SpLR prefers L2 over dropout.

4. **CIFAR-10 baseline**  
   Repeat the same comparison on CIFAR-10 to see if gains shrink on easier data.

5. **Deeper architectures (ResNet-style)**  
   Test whether SpLR’s advantage grows or shrinks in deeper residual networks.

This benchmark (A-3) serves as a strong indicator that SpLR is a serious activation candidate for fine-grained vision, not just a toy function.


# THE Benchmarak USED

In [ ]:
# ============================================================
# I-0 CIFAR-100: Mish vs SpLR (Golden Init), dropout = 0.10
# ============================================================

import os, random, math, time
from typing import List, Dict, Any, Tuple

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split

from torchvision import datasets, transforms

# ============================================================
# 0) DEVICE + SEEDING
# ============================================================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PIN_MEMORY = (DEVICE == "cuda")
print("DEVICE:", DEVICE)

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# ============================================================
# 1) HYPERPARAMS FOR BENCHMARK
# ============================================================

ACTIVATIONS = ["mish", "SpLR"]
SEEDS = [42, 1337, 777]

BATCH_SIZE = 128
EPOCHS = 25        # bump to 30 if you want more training
LR = 1e-3
P_DROPOUT = 0.10

NUM_CLASSES = 100

# ============================================================
# 2) DATA: CIFAR-100
# ============================================================

# Standard CIFAR-100 normalization
CIFAR100_MEAN = [0.5071, 0.4867, 0.4408]
CIFAR100_STD  = [0.2675, 0.2565, 0.2761]

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])

root = "./data"
train_full = datasets.CIFAR100(root=root, train=True,  download=True, transform=train_transform)
test_set   = datasets.CIFAR100(root=root, train=False, download=True, transform=test_transform)

val_size = 5000
train_size = len(train_full) - val_size
train_set, val_set = random_split(train_full, [train_size, val_size])

print(f"Train size: {len(train_set)}, Val size: {len(val_set)}, Test size: {len(test_set)}")

def make_loaders() -> Tuple[DataLoader, DataLoader, DataLoader]:
    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=2, pin_memory=PIN_MEMORY)
    val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=2, pin_memory=PIN_MEMORY)
    test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=2, pin_memory=PIN_MEMORY)
    return train_loader, val_loader, test_loader

# ============================================================
# 3) SpLR ACTIVATION (Golden Init)
# ============================================================

class SpLR_2d(nn.Module):
    """
    SpLR for Conv2d outputs.
    y = x + alpha * x * exp(-beta * x^2)

    alpha_raw: (1, C, 1, 1), init = 0.182
        alpha = 2 * tanh(alpha_raw)   (~0.36 at init), trainable per-channel
    beta_raw: scalar, init = -1.35
        beta = 0.01 + softplus(beta_raw)  (~0.24 at init), trainable, shared per block
    """
    def __init__(self, num_channels: int):
        super().__init__()
        self.alpha_raw = nn.Parameter(torch.full((1, num_channels, 1, 1), 0.182))
        self.beta_raw  = nn.Parameter(torch.tensor(-1.35))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        alpha = 2.0 * torch.tanh(self.alpha_raw)
        beta  = 0.01 + F.softplus(self.beta_raw)
        return x + alpha * x * torch.exp(-beta * x * x)

class SpLR_1d(nn.Module):
    """
    SpLR for Linear outputs.
    alpha_raw: (1, F)
    beta_raw: scalar
    """
    def __init__(self, num_features: int):
        super().__init__()
        self.alpha_raw = nn.Parameter(torch.full((1, num_features), 0.182))
        self.beta_raw  = nn.Parameter(torch.tensor(-1.35))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        alpha = 2.0 * torch.tanh(self.alpha_raw)
        beta  = 0.01 + F.softplus(self.beta_raw)
        return x + alpha * x * torch.exp(-beta * x * x)

# ============================================================
# 4) MODEL: CIFAR-100 CNN with pluggable activation
# ============================================================

class Cifar100Net(nn.Module):
    """
    Simple CNN for CIFAR-100 with pluggable nonlinearity:
    - activ_kind = "mish"  -> Mish activations
    - activ_kind = "SpLR"  -> SpLR activations (Golden Init)
    Dropout p is applied after each activation.
    """
    def __init__(self, activ_kind: str, p_dropout: float):
        super().__init__()

        assert activ_kind in ["mish", "SpLR"]
        self.activ_kind = activ_kind
        self.p = p_dropout

        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.bn1   = nn.BatchNorm2d(64)
        self.spv1  = SpLR_2d(64) if activ_kind == "SpLR" else None
        self.do1   = nn.Dropout2d(p_dropout)
        self.pool1 = nn.MaxPool2d(2)  # 32 -> 16

        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn2   = nn.BatchNorm2d(128)
        self.spv2  = SpLR_2d(128) if activ_kind == "SpLR" else None
        self.do2   = nn.Dropout2d(p_dropout)
        self.pool2 = nn.MaxPool2d(2)  # 16 -> 8

        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn3   = nn.BatchNorm2d(256)
        self.spv3  = SpLR_2d(256) if activ_kind == "SpLR" else None
        self.do3   = nn.Dropout2d(p_dropout)
        self.pool3 = nn.MaxPool2d(2)  # 8 -> 4

        self.flat_dim = 256 * 4 * 4

        self.fc1    = nn.Linear(self.flat_dim, 512)
        self.spv_fc = SpLR_1d(512) if activ_kind == "SpLR" else None
        self.do_fc  = nn.Dropout(p_dropout)
        self.fc_out = nn.Linear(512, NUM_CLASSES)

    def nonlinearity_conv(self, x: torch.Tensor, block: int) -> torch.Tensor:
        if self.activ_kind == "mish":
            return F.mish(x)
        else:
            if block == 1:
                return self.spv1(x)
            elif block == 2:
                return self.spv2(x)
            elif block == 3:
                return self.spv3(x)
            else:
                raise ValueError("Invalid block index")

    def nonlinearity_fc(self, x: torch.Tensor) -> torch.Tensor:
        if self.activ_kind == "mish":
            return F.mish(x)
        else:
            return self.spv_fc(x)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Block 1
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.nonlinearity_conv(x, block=1)
        x = self.do1(x)
        x = self.pool1(x)

        # Block 2
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.nonlinearity_conv(x, block=2)
        x = self.do2(x)
        x = self.pool2(x)

        # Block 3
        x = self.conv3(x)
        x = self.bn3(x)
        x = self.nonlinearity_conv(x, block=3)
        x = self.do3(x)
        x = self.pool3(x)

        # Flatten
        x = x.view(x.size(0), -1)

        # FC
        x = self.fc1(x)
        x = self.nonlinearity_fc(x)
        x = self.do_fc(x)
        x = self.fc_out(x)

        return x

def make_model(activ_kind: str) -> nn.Module:
    model = Cifar100Net(activ_kind=activ_kind, p_dropout=P_DROPOUT)
    return model.to(DEVICE)

# ============================================================
# 5) TRAIN / EVAL LOOPS
# ============================================================

def train_one_epoch(model: nn.Module,
                    loader: DataLoader,
                    optimizer: torch.optim.Optimizer,
                    criterion: nn.Module) -> Tuple[float, float]:
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        total += labels.size(0)
        correct += (preds == labels).sum().item()

    return running_loss / total, correct / total


def evaluate(model: nn.Module,
             loader: DataLoader,
             criterion: nn.Module) -> Tuple[float, float]:
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()

    return running_loss / total, correct / total

def train_and_eval_model(activ_kind: str,
                         seed: int,
                         epochs: int = EPOCHS) -> Dict[str, Any]:
    set_seed(seed)
    train_loader, val_loader, test_loader = make_loaders()

    model = make_model(activ_kind)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    best_val_acc = 0.0
    best_test_acc_at_best_val = 0.0

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_acc     = evaluate(model, val_loader, criterion)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            _, test_acc = evaluate(model, test_loader, criterion)
            best_test_acc_at_best_val = test_acc

        t1 = time.time()
        print(f"[{activ_kind.upper()} | seed={seed}] "
              f"Epoch {epoch:02d}/{epochs:02d} | "
              f"tr_loss={train_loss:.4f} tr_acc={train_acc*100:.2f}% | "
              f"val_loss={val_loss:.44f} val_acc={val_acc*100:.2f}% | "
              f"time={t1-t0:.1f}s")

    return {
        "activation": activ_kind,
        "seed": seed,
        "best_val_acc": best_val_acc,
        "best_test_acc_at_best_val": best_test_acc_at_best_val,
    }

# ============================================================
# 6) OUTER BENCHMARK LOOP: Mish vs SpLR
# ============================================================

all_results: List[Dict[str, Any]] = []

for activ in ACTIVATIONS:
    for seed in SEEDS:
        print("=" * 70)
        print(f"Running CIFAR-100 with activation={activ}, dropout={P_DROPOUT}, seed={seed}")
        print("=" * 70)
        result = train_and_eval_model(activ_kind=activ, seed=seed, epochs=EPOCHS)
        all_results.append(result)

results_df = pd.DataFrame(all_results)
results_df.to_csv("cifar100_mish_vs_SpLR_raw.csv", index=False)
print("\nRaw results:")
print(results_df)

summary = results_df.groupby("activation").agg(
    mean_val_acc=("best_val_acc", "mean"),
    std_val_acc=("best_val_acc", "std"),
    mean_test_acc=("best_test_acc_at_best_val", "mean"),
    std_test_acc=("best_test_acc_at_best_val", "std"),
).reset_index()

summary.to_csv("cifar100_mish_vs_SpLR_summary.csv", index=False)
print("\nSummary:")
print(summary)
